# What you'll learn

- Why attention dilution produces inconsistent analysis depth across files in single-pass reviews
- How multi-pass architecture (per-item + cross-item) solves the structural attention allocation problem
- The difference between fixed sequential pipelines and dynamic adaptive decomposition
- Why batching without a cross-file integration pass still misses cross-cutting issues
- How to identify attention dilution artefacts: same pattern flagged in one file, approved in another

In [1]:
from pathlib import Path

SOURCE_EXTENSIONS = {".js", ".ts", ".jsx", ".tsx"}
MIN_FILES = 14


def load_source_files(directory):
    """Read every JS/TS file in a folder. Returns a list of {path, content}."""
    root = Path(directory)
    if not root.is_dir():
        raise ValueError(f"Not a directory: {directory}")

    files = [
        {"path": str(p.relative_to(root)), "content": p.read_text()}
        for p in sorted(root.rglob("*"))
        if p.is_file() and p.suffix in SOURCE_EXTENSIONS
    ]

    if len(files) < MIN_FILES:
        raise ValueError(f"Need at least {MIN_FILES} source files, found {len(files)}")
    return files


def format_for_review(files):
    """Join the files into one text block. Each file has its path and numbered lines."""
    blocks = []
    for f in files:
        numbered = "\n".join(
            f"{n}: {line}" for n, line in enumerate(f["content"].splitlines(), 1)
        )
        blocks.append(f"=== {f['path']} ===\n{numbered}")
    return "\n\n".join(blocks)

In [2]:
files = load_source_files("bookshop_src")
print(f"Loaded {len(files)} files:")
for f in files:
    print(f"- {f['path']} ({len(f['content'])} chars)")

Loaded 14 files:
- auth.ts (849 chars)
- authors.js (955 chars)
- books.js (1000 chars)
- cart.js (956 chars)
- checkout.ts (992 chars)
- db.js (773 chars)
- inventory.ts (917 chars)
- orders.js (1203 chars)
- payments.ts (1097 chars)
- reviews.js (870 chars)
- search.js (914 chars)
- shipping.js (939 chars)
- users.js (928 chars)
- utils.js (861 chars)


## Baseline: single-pass review

One request with all 14 files inside. This is the baseline. If attention dilution is real, the
feedback gets thinner the further down the prompt a file sits: early files get line numbers and bug
details, later files get one short line or nothing.

In [3]:
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
model = "claude-haiku-4-5"

In [4]:
import json

RESULTS_DIR = Path("review_results")

REVIEW_PROMPT = """Review all of the files below in one pass.
For each file, write a heading "## <file name>", then list the bugs you find.
Each bug is one bullet: the line number and what is wrong.

{code}"""


def single_pass_review(files):
    """Send all files in ONE request. Save the result to a JSON file and return it."""
    response = client.messages.create(
        model=model,
        max_tokens=8000,
        messages=[
            {"role": "user", "content": REVIEW_PROMPT.format(code=format_for_review(files))}
        ],
    )
    result = {
        "approach": "single_pass",
        "files": [f["path"] for f in files],
        "review": response.content[0].text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }
    RESULTS_DIR.mkdir(exist_ok=True)
    (RESULTS_DIR / "single_pass.json").write_text(json.dumps(result, indent=2))
    return result

In [5]:
single_pass_result = single_pass_review(files)
print(single_pass_result["review"])

## auth.ts

- Line 11: Using MD5 for password hashing is cryptographically weak and insecure
- Line 25: `logout()` doesn't actually remove the token from sessions; it just returns true
- Line 29: `Math.random()` is not cryptographically secure for generating reset tokens
- Line 33: Logic error in `isAdmin()` - condition `user.role == "admin" || "superadmin"` always returns true; should be `user.role === "admin" || user.role === "superadmin"`

## authors.js

- Line 10: `getAuthorBio()` will crash if `author` is null or `author.bio` is null/undefined
- Line 23: Type comparison bug - `books.length == "0"` compares number to string; should be `=== 0`
- Line 28: Missing `DESC` in ORDER BY clause - query returns authors with least books first instead of most

## books.js

- Line 4: SQL injection vulnerability - `genre` parameter not parameterized
- Line 8: SQL injection vulnerability - `id` parameter not parameterized

## cart.js

- Line 20: `updateQuantity()` doesn't handle case where item 

In [6]:
import re

# Bugs planted in each file (our own count). Used to see what the review missed.
PLANTED_BUGS = {
    "auth.ts": 4, "authors.js": 3, "books.js": 4, "cart.js": 3, "checkout.ts": 3,
    "db.js": 4, "inventory.ts": 3, "orders.js": 3, "payments.ts": 4, "reviews.js": 3,
    "search.js": 3, "shipping.js": 3, "users.js": 4, "utils.js": 3,
}


def breakdown(files, review_text):
    """For each file (in prompt order): bugs planted, bugs reported, words of feedback."""
    sections = {}
    for part in re.split(r"^## ", review_text, flags=re.MULTILINE)[1:]:
        name, _, body = part.partition("\n")
        sections[name.strip()] = body

    print(f"{'#':>2}  {'file':<14}{'planted':>8}{'reported':>9}{'words':>7}")
    for i, f in enumerate(files, 1):
        body = sections.get(f["path"])
        if body is None:
            print(f"{i:>2}  {f['path']:<14}{PLANTED_BUGS[f['path']]:>8}{'MISSING':>9}{0:>7}")
            continue
        reported = sum(1 for line in body.splitlines() if line.lstrip().startswith("- "))
        print(f"{i:>2}  {f['path']:<14}{PLANTED_BUGS[f['path']]:>8}{reported:>9}{len(body.split()):>7}")


breakdown(files, single_pass_result["review"])

 #  file           planted reported  words
 1  auth.ts              4        4     66
 2  authors.js           3        3     52
 3  books.js             4        2     22
 4  cart.js              3        2     34
 5  checkout.ts          3        1     18
 6  db.js                4        1     16
 7  inventory.ts         3        1     14
 8  orders.js            3        1     14
 9  payments.ts          4        3     40
10  reviews.js           3        2     32
11  search.js            3        2     26
12  shipping.js          3        3     49
13  users.js             4        1     13
14  utils.js             3        3     49


## Pass 1: per-file review

Same review, but one request per file. Each file gets the model's full attention, and each answer
comes back in a fixed shape: bug count, and for every bug a line number and a severity.

In [7]:
SEVERITIES = ["high", "medium", "low"]

# The model must answer by calling this tool, so every file comes back in the same shape.
REPORT_TOOL = {
    "name": "report_review",
    "description": "Report the bugs found in one file.",
    "input_schema": {
        "type": "object",
        "required": ["bugs"],
        "properties": {
            "bugs": {
                "type": "array",
                "items": {
                    "type": "object",
                    "required": ["line", "severity", "description"],
                    "properties": {
                        "line": {"type": "integer", "description": "Line number of the bug"},
                        "severity": {"type": "string", "enum": SEVERITIES},
                        "description": {"type": "string", "description": "What is wrong"},
                    },
                },
            }
        },
    },
}

# Same instructions as REVIEW_PROMPT, for one file.
PER_FILE_PROMPT = """Review the file below.
List the bugs you find.
Each bug has the line number, a severity and what is wrong.

{code}"""


def review_one_file(file):
    """Send ONE file. Returns {file, bug_count, bugs}."""
    response = client.messages.create(
        model=model,
        max_tokens=2000,
        tools=[REPORT_TOOL],
        tool_choice={"type": "tool", "name": "report_review"},
        messages=[
            {"role": "user", "content": PER_FILE_PROMPT.format(code=format_for_review([file]))}
        ],
    )
    tool_call = next(block for block in response.content if block.type == "tool_use")
    bugs = tool_call.input["bugs"]
    return {"file": file["path"], "bug_count": len(bugs), "bugs": bugs}


def per_file_review(files):
    """Review every file in its own request. Save all results to a JSON file."""
    results = [review_one_file(f) for f in files]
    RESULTS_DIR.mkdir(exist_ok=True)
    (RESULTS_DIR / "per_file.json").write_text(json.dumps(results, indent=2))
    return results

In [8]:
per_file_results = per_file_review(files)

for r in per_file_results:
    print(f"\n{r['file']} - {r['bug_count']} bugs")
    for b in r["bugs"]:
        print(f"  line {b['line']} [{b['severity']}] {b['description']}")


auth.ts - 6 bugs
  line 11 [high] MD5 is used for password hashing which is cryptographically broken. Should use a proper password hashing algorithm like bcrypt, scrypt, or argon2.
  line 25 [high] logout() function doesn't actually remove the session token. It should call sessions.delete(token) to invalidate the session.
  line 29 [high] createResetToken() uses Math.random() which is not cryptographically secure. Should use crypto.randomBytes() like in the login function.
  line 33 [high] Logical error in isAdmin() function. The expression 'user.role == "admin" || "superadmin"' will always evaluate to true because "superadmin" is a truthy string. Should be 'user.role === "admin" || user.role === "superadmin"'.
  line 4 [medium] Sessions Map stores user IDs as values but doesn't have expiration/TTL. Sessions will persist indefinitely in memory, potentially causing memory leaks and security issues.
  line 33 [medium] Using loose equality (==) instead of strict equality (===) for string

In [9]:
def per_file_breakdown(results):
    """For each file: bugs planted, bugs reported, and how many of each severity."""
    print(f"{'#':>2}  {'file':<14}{'planted':>8}{'reported':>9}{'high':>6}{'medium':>7}{'low':>5}")
    for i, r in enumerate(results, 1):
        counts = {s: sum(1 for b in r["bugs"] if b["severity"] == s) for s in SEVERITIES}
        print(
            f"{i:>2}  {r['file']:<14}{PLANTED_BUGS[r['file']]:>8}{r['bug_count']:>9}"
            f"{counts['high']:>6}{counts['medium']:>7}{counts['low']:>5}"
        )


per_file_breakdown(per_file_results)

 #  file           planted reported  high medium  low
 1  auth.ts              4        6     4      2    0
 2  authors.js           3        3     3      0    0
 3  books.js             4        3     2      1    0
 4  cart.js              3        3     3      0    0
 5  checkout.ts          3        3     2      1    0
 6  db.js                4        3     2      1    0
 7  inventory.ts         3        5     2      3    0
 8  orders.js            3        3     1      2    0
 9  payments.ts          4        4     2      2    0
10  reviews.js           3        3     1      2    0
11  search.js            3        4     2      2    0
12  shipping.js          3        3     1      2    0
13  users.js             4        3     1      2    0
14  utils.js             3        3     3      0    0


## Pass 2: cross-file integration

A separate request. It gets the per-file findings (so it does not repeat them) and the code (so it
can follow data from one file to another). It looks only at problems between files.

In [10]:
CROSS_CATEGORIES = ["data_flow", "api_consistency", "module_interaction", "pattern_consistency"]

CROSS_FILE_TOOL = {
    "name": "report_cross_file_issues",
    "description": "Report issues that involve two or more files.",
    "input_schema": {
        "type": "object",
        "required": ["issues"],
        "properties": {
            "issues": {
                "type": "array",
                "items": {
                    "type": "object",
                    "required": ["category", "severity", "locations", "description"],
                    "properties": {
                        "category": {"type": "string", "enum": CROSS_CATEGORIES},
                        "severity": {"type": "string", "enum": SEVERITIES},
                        "locations": {
                            "type": "array",
                            "minItems": 2,
                            "items": {
                                "type": "object",
                                "required": ["file", "line"],
                                "properties": {
                                    "file": {"type": "string"},
                                    "line": {"type": "integer"},
                                },
                            },
                        },
                        "description": {"type": "string", "description": "What does not fit together"},
                    },
                },
            }
        },
    },
}

CROSS_FILE_PROMPT = """Each file was already reviewed on its own. The findings are below.
Look only at problems BETWEEN files. Every issue must point to places in at least two
DIFFERENT files (file and line). Do not report a bug that lives inside one file, even if it is
listed below. Only report what you can see in the code. Do not guess.

Categories:
- data_flow: a value changes shape, name or unit as it moves from one module to another
- api_consistency: a function is called in a way that does not match how it is defined
- module_interaction: modules that should work together but skip a step or do it in the wrong order
- pattern_consistency: the same thing is done one way in some files and another way in others

PER-FILE FINDINGS:
{summaries}

SOURCE CODE:
{code}"""


def summarise_per_file(per_file_results):
    """Turn the per-file results into short text lines for the next pass."""
    lines = []
    for r in per_file_results:
        lines.append(f"{r['file']} ({r['bug_count']} bugs)")
        for b in r["bugs"]:
            lines.append(f"  line {b['line']} [{b['severity']}] {b['description']}")
    return "\n".join(lines)


def cross_file_review(files, per_file_results):
    """One request that gets the per-file findings plus the code. Saves a JSON file."""
    prompt = CROSS_FILE_PROMPT.format(
        summaries=summarise_per_file(per_file_results),
        code=format_for_review(files),
    )
    response = client.messages.create(
        model=model,
        max_tokens=4000,
        tools=[CROSS_FILE_TOOL],
        tool_choice={"type": "tool", "name": "report_cross_file_issues"},
        messages=[{"role": "user", "content": prompt}],
    )
    tool_call = next(block for block in response.content if block.type == "tool_use")
    issues = tool_call.input["issues"]

    # The schema cannot force "two different files", so we check it here.
    kept = [i for i in issues if len({loc["file"] for loc in i["locations"]}) >= 2]

    result = {"approach": "cross_file", "issues": kept, "dropped_same_file": len(issues) - len(kept)}
    RESULTS_DIR.mkdir(exist_ok=True)
    (RESULTS_DIR / "cross_file.json").write_text(json.dumps(result, indent=2))
    return result

In [11]:
cross_file_result = cross_file_review(files, per_file_results)

print(f"{len(cross_file_result['issues'])} cross-file issues")
print(f"({cross_file_result['dropped_same_file']} dropped: all places were in one file)\n")
for issue in cross_file_result["issues"]:
    places = ", ".join(f"{loc['file']}:{loc['line']}" for loc in issue["locations"])
    print(f"[{issue['severity']}] {issue['category']} - {places}")
    print(f"  {issue['description']}\n")

print("By category:")
for category in CROSS_CATEGORIES:
    count = sum(1 for i in cross_file_result["issues"] if i["category"] == category)
    print(f"  {category:<20}{count}")

11 cross-file issues
(15 dropped: all places were in one file)

[high] data_flow - checkout.ts:20, shipping.js:4
  checkout.ts assigns cartTotal() result (a number representing price) to variable 'items' (line 20), then passes it to shippingCost() as the second parameter (line 21). shippingCost() in shipping.js expects 'totalCents' (line 4), which is a price amount, but the parameter name and usage in checkout.ts is misleading since 'items' suggests an array of cart items, not a price total.

[high] api_consistency - books.js:4, users.js:21
  books.js listBooks() uses string interpolation for SQL injection vulnerable query (line 4), while other files like authors.js (line 4), search.js (line 19), and orders.js (line 25) mix parameterized and non-parameterized queries. users.js updateEmail() (line 21) also uses string interpolation inconsistently with other update functions.

[high] module_interaction - db.js:12, inventory.ts:23
  db.js transaction() function doesn't rollback on error (

In [12]:
def combined_breakdown(per_file_results, cross_file_result):
    """Both passes side by side. A cross-file issue counts once for every file it points to."""
    print(f"{'#':>2}  {'file':<14}{'planted':>8}{'per-file':>9}{'cross-file':>11}{'combined':>9}")
    totals = [0, 0, 0, 0]
    for i, r in enumerate(per_file_results, 1):
        name = r["file"]
        cross = sum(
            1
            for issue in cross_file_result["issues"]
            if name in {loc["file"] for loc in issue["locations"]}
        )
        row = [PLANTED_BUGS[name], r["bug_count"], cross, r["bug_count"] + cross]
        totals = [t + v for t, v in zip(totals, row)]
        print(f"{i:>2}  {name:<14}{row[0]:>8}{row[1]:>9}{row[2]:>11}{row[3]:>9}")
    print(f"{'':>2}  {'total':<14}{totals[0]:>8}{totals[1]:>9}{totals[2]:>11}{totals[3]:>9}")


combined_breakdown(per_file_results, cross_file_result)

 #  file           planted per-file cross-file combined
 1  auth.ts              4        6          3        9
 2  authors.js           3        3          1        4
 3  books.js             4        3          2        5
 4  cart.js              3        3          1        4
 5  checkout.ts          3        3          3        6
 6  db.js                4        3          3        6
 7  inventory.ts         3        5          1        6
 8  orders.js            3        3          0        3
 9  payments.ts          4        4          0        4
10  reviews.js           3        3          0        3
11  search.js            3        4          1        5
12  shipping.js          3        3          2        5
13  users.js             4        3          3        6
14  utils.js             3        3          2        5
    total               47       49         22       71


## Compare: single-pass vs multi-pass

What to look for:
- Multi-pass finds more issues in total.
- Multi-pass counts stay level from the first file to the last. Single-pass counts drop.
- Some issues (between files) are only found by the integration pass.

In [ ]:
def single_pass_bugs(review_text):
    """Read the single-pass review. Returns {file: [line numbers]}, one number per bullet."""
    found = {}
    for part in re.split(r"^## ", review_text, flags=re.MULTILINE)[1:]:
        name, _, body = part.partition("\n")
        found[name.strip()] = [
            (int(m.group(1)) if (m := re.search(r"[Ll]ines?\s*(\d+)", line)) else None)
            for line in body.splitlines()
            if line.lstrip().startswith("- ")
        ]
    return found


def single_pass_saw_issue(review_text, issue):
    """Rough check: did the single-pass review of one file mention another file of the issue?"""
    names = {loc["file"] for loc in issue["locations"]}
    for part in re.split(r"^## ", review_text, flags=re.MULTILINE)[1:]:
        name, _, body = part.partition("\n")
        if name.strip() in names and any(other in body for other in names - {name.strip()}):
            return True
    return False


def compare_results(files, single_pass_result, per_file_results, cross_file_result):
    review = single_pass_result["review"]
    single = single_pass_bugs(review)
    multi = {r["file"]: r for r in per_file_results}

    # Each cross-file issue is counted once, at the first file it points to.
    cross_at = {f["path"]: 0 for f in files}
    for issue in cross_file_result["issues"]:
        cross_at[issue["locations"][0]["file"]] += 1

    # ---- Table 1: counts per file ----
    print("TABLE 1 - issues per file")
    print(f"{'#':>2}  {'file':<14}{'planted':>8}{'single':>8}{'multi':>7}   (multi = per-file + cross-file)")
    singles, multis, per_file_counts = [], [], []
    for i, f in enumerate(files, 1):
        name = f["path"]
        s = len(single.get(name, []))
        p = multi[name]["bug_count"]
        singles.append(s)
        per_file_counts.append(p)
        multis.append(p + cross_at[name])
        print(f"{i:>2}  {name:<14}{PLANTED_BUGS[name]:>8}{s:>8}{multis[-1]:>7}")
    print(f"{'':>2}  {'total':<14}{sum(PLANTED_BUGS.values()):>8}{sum(singles):>8}{sum(multis):>7}")

    # ---- Consistency: early files vs late files (per-file pass only, no cross-file) ----
    half = len(files) // 2
    print("\nCONSISTENCY - average issues per file (multi-pass = per-file pass only)")
    print(f"{'':<24}{'first 7':>9}{'last 7':>8}{'lowest':>8}")
    for label, counts in (("single-pass", singles), ("multi-pass (per-file)", per_file_counts)):
        print(
            f"{label:<24}{sum(counts[:half]) / half:>9.1f}"
            f"{sum(counts[-half:]) / half:>8.1f}{min(counts):>8}"
        )

    # ---- Table 2: which issues each approach caught (same file and same line) ----
    print("\nTABLE 2 - per-file bugs: single-pass vs multi-pass (same file and line)")
    print(f"{'#':>2}  {'file':<14}{'both':>6}{'multi only':>11}{'single only':>12}")
    totals = [0, 0, 0]
    for i, f in enumerate(files, 1):
        name = f["path"]
        s_lines = {n for n in single.get(name, []) if n is not None}
        m_lines = {b["line"] for b in multi[name]["bugs"]}
        row = [len(s_lines & m_lines), len(m_lines - s_lines), len(s_lines - m_lines)]
        totals = [t + v for t, v in zip(totals, row)]
        print(f"{i:>2}  {name:<14}{row[0]:>6}{row[1]:>11}{row[2]:>12}")
    print(f"{'':>2}  {'total':<14}{totals[0]:>6}{totals[1]:>11}{totals[2]:>12}")

    # ---- Cross-file issues: did the single-pass review see them? ----
    print("\nCROSS-FILE ISSUES - seen by the single-pass review?")
    only_integration = 0
    for issue in cross_file_result["issues"]:
        seen = single_pass_saw_issue(review, issue)
        only_integration += not seen
        places = ", ".join(f"{loc['file']}:{loc['line']}" for loc in issue["locations"])
        print(f"  {'single-pass saw it' if seen else 'ONLY integration pass':<22} {issue['category']:<20}{places}")
    print(f"\n{only_integration} of {len(cross_file_result['issues'])} cross-file issues were caught only by the integration pass.")


compare_results(files, single_pass_result, per_file_results, cross_file_result)

## Attention dilution artefacts

The same code pattern, flagged in one file and left alone in another. If the single-pass review does
this, the model is not judging the code, it is running out of attention. The multi-pass review should
treat the same pattern the same way every time.

In [ ]:
# Code patterns that appear in several files. Found with a simple text search, not with the model.
PATTERNS = {
    "sql_built_from_text": (
        r"`[^`]*(SELECT|INSERT|UPDATE|DELETE)[^`]*\$\{"
        r"|(SELECT|INSERT|UPDATE|DELETE)[^\n]*\"\s*\+\s*\w",
        "SQL query built by joining text (injection risk)",
    ),
    "unchecked_first_row": (
        r"rows\[0\]\.\w+",
        "rows[0].something used without checking rows is empty",
    ),
    "return_first_row": (
        r"return rows\[0\];",
        "return rows[0] with no check that a row exists",
    ),
    "order_by_no_direction": (
        r"ORDER BY \w+\"",
        "ORDER BY with no ASC/DESC",
    ),
    "loose_equality": (
        r"(?<![=!<>])==(?!=)",
        "== used instead of ===",
    ),
}
LINE_TOLERANCE = 1  # a bug on line 8 counts for a pattern on line 7 to 9


def find_pattern_lines(files, regex):
    """Every (file, line) whose text matches the pattern."""
    return [
        (f["path"], n)
        for f in files
        for n, line in enumerate(f["content"].splitlines(), 1)
        if re.search(regex, line)
    ]


def is_flagged(reported_lines, line):
    return any(abs(r - line) <= LINE_TOLERANCE for r in reported_lines if r is not None)


def find_contradictions(record, approach):
    """Patterns that one approach flagged in some places and left alone in others."""
    contradictions = []
    for p in record:
        flagged = [f"{r['file']}:{r['line']}" for r in p["occurrences"] if r[approach]]
        approved = [f"{r['file']}:{r['line']}" for r in p["occurrences"] if not r[approach]]
        if flagged and approved:
            contradictions.append(
                f"Pattern \"{p['pattern']}\" flagged in [{', '.join(flagged)}]"
                f" but approved in [{', '.join(approved)}]"
            )
    return contradictions


def find_dilution_artefacts(files, single_pass_result, per_file_results):
    """Same pattern: flagged in some files, left alone in others. Saves a JSON file."""
    single = single_pass_bugs(single_pass_result["review"])
    multi = {r["file"]: [b["line"] for b in r["bugs"]] for r in per_file_results}

    record = []
    for name, (regex, label) in PATTERNS.items():
        rows = [
            {
                "file": file,
                "line": line,
                "single_pass": is_flagged(single.get(file, []), line),
                "multi_pass": is_flagged(multi[file], line),
            }
            for file, line in find_pattern_lines(files, regex)
        ]
        record.append({"pattern": name, "label": label, "occurrences": rows})

    for p in record:
        print(f"\n{p['pattern']} - {p['label']}")
        print(f"  {'where':<20}{'single-pass':<14}{'multi-pass'}")
        for r in p["occurrences"]:
            single_text = "flagged" if r["single_pass"] else "NOT flagged"
            multi_text = "flagged" if r["multi_pass"] else "NOT flagged"
            print(f"  {r['file'] + ':' + str(r['line']):<20}{single_text:<14}{multi_text}")

    contradictions = {
        "single_pass": find_contradictions(record, "single_pass"),
        "multi_pass": find_contradictions(record, "multi_pass"),
    }
    for approach, items in contradictions.items():
        print(f"\nCONTRADICTIONS - {approach} ({len(items)})")
        for text in items:
            print(f"  {text}")

    RESULTS_DIR.mkdir(exist_ok=True)
    out = {"patterns": record, "contradictions": contradictions}
    (RESULTS_DIR / "dilution_artefacts.json").write_text(json.dumps(out, indent=2))
    return out


artefacts = find_dilution_artefacts(files, single_pass_result, per_file_results)